<a href="https://colab.research.google.com/github/leoncana/CreativeHub-DataEngineeringModule1/blob/main/homework/hw04_spark_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 4 — Spark basics

About 45 minutes. You explore Wavely's **users** and **artists** tables with Spark, using schemas, filters, sorting and `groupBy` from session 4.

`users.csv` columns: `user_id, signup_date, country, plan, birth_year` (birth year is optional, so sometimes empty)

In [3]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


In [4]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

Spark 4.0.5 is ready


### Exercise 1 — Load with a schema
Read `DATA / "reference" / "users.csv"` into `users` with an explicit schema.
Use `DATE` for `signup_date` and `INT` for `birth_year`.

In [6]:
# YOUR CODE HERE

users = spark.read.csv(str(DATA/ "reference" / "users.csv"), header=True, schema='user_id STRING, signup_date DATE, country STRING, plan STRING, birth_year INT' )

users.printSchema()
users.show(5)
print(users.count())  # expected: 800

root
 |-- user_id: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- plan: string (nullable = true)
 |-- birth_year: integer (nullable = true)

+-------+-----------+-------+-------+----------+
|user_id|signup_date|country|   plan|birth_year|
+-------+-----------+-------+-------+----------+
|  U0001| 2025-08-31|     SK|   free|      2001|
|  U0002| 2019-03-26|     SK|premium|      1999|
|  U0003| 2019-07-12|     CZ|   free|      1992|
|  U0004| 2025-10-03|     SK|   free|      1997|
|  U0005| 2019-11-07|     CZ|   free|      1988|
+-------+-----------+-------+-------+----------+
only showing top 5 rows
800


### Exercise 2 — Free vs premium
Count users per `plan`, and users per `country`, each sorted by count (largest first).

In [18]:
# YOUR CODE HERE

users.groupBy("plan").count().orderBy(F.desc("count")).show()
users.groupBy("country").count().orderBy(F.desc('count')).show()


+-------+-----+
|   plan|count|
+-------+-----+
|   free|  456|
|premium|  344|
+-------+-----+

+-------+-----+
|country|count|
+-------+-----+
|     AT|  241|
|     SK|  130|
|     CZ|  121|
|     DE|  119|
|     HU|   99|
|     PL|   90|
+-------+-----+



### Exercise 3 — Missing birth years
How many users have no `birth_year`? Store it in `n_missing`.

In [19]:
# YOUR CODE HERE
n_missing = users.filter(F.col("birth_year").isNull()).count()

print(n_missing)

26


### Exercise 4 — Young listeners in Austria
1. Add a column `age` = `2026 - birth_year`.
2. Keep users from `AT` aged **under 25**.
3. Show `user_id`, `plan`, `age`, youngest first, and store the number of such users in `n_young_at`.

In [30]:
# YOUR CODE HERE
n_young_at = (
    users
    .withColumn('age', 2026 - F.col('birth_year')).orderBy('age')
    .filter((F.col('country')=='AT') & (F.col('age')<25))
    .select('user_id', 'plan', 'age')
)
n_young_at.show()
print(n_young_at.count())

+-------+-------+---+
|user_id|   plan|age|
+-------+-------+---+
|  U0110|   free| 16|
|  U0139|premium| 16|
|  U0257|   free| 16|
|  U0367|   free| 16|
|  U0437|premium| 16|
|  U0553|   free| 16|
|  U0583|   free| 16|
|  U0605|   free| 16|
|  U0622|premium| 16|
|  U0660|   free| 16|
|  U0733|   free| 16|
|  U0794|   free| 16|
|  U0202|premium| 17|
|  U0536|   free| 17|
|  U0676|   free| 17|
|  U0702|premium| 17|
|  U0014|   free| 18|
|  U0313|   free| 19|
|  U0415|premium| 19|
|  U0535|premium| 19|
+-------+-------+---+
only showing top 20 rows
61


### Exercise 5 — Sign-ups per year
`F.year(F.col("signup_date"))` extracts the year from a date. Count sign-ups per year, sorted by year.
Which year had the most sign-ups?

In [37]:
# YOUR CODE HERE
#users.show()

users.groupBy(F.year(F.col("signup_date")).alias('year of signup')).count().orderBy(F.year(F.col("signup_date"))).show()

+--------------+-----+
|year of signup|count|
+--------------+-----+
|          2019|  100|
|          2020|  110|
|          2021|  117|
|          2022|  108|
|          2023|  104|
|          2024|  100|
|          2025|   92|
|          2026|   69|
+--------------+-----+



### Exercise 6 — Artists
Load `artists.csv` (`artist_id, artist_name, country, genre`) and show the artists that are **not** from the US or GB,
sorted by name.

In [40]:
# YOUR CODE HERE
artists = spark.read.csv(str(DATA/ "reference" / "artists.csv"), header=True, inferSchema=True)
artists.show()

artists.filter(~F.col('country').isin('US', 'GB')).orderBy('artist_name').show()

+---------+----------------+-------+----------+
|artist_id|     artist_name|country|     genre|
+---------+----------------+-------+----------+
|      A01|    Taylor Swift|     US|       pop|
|      A02|        Dua Lipa|     GB|       pop|
|      A03|      The Weeknd|     CA|       pop|
|      A04|   Billie Eilish|     US|       pop|
|      A05|      Ed Sheeran|     GB|       pop|
|      A06|           Adele|     GB|       pop|
|      A07|       Bad Bunny|     PR|     latin|
|      A08|         Rosalía|     ES|     latin|
|      A09|  Kendrick Lamar|     US|   hip-hop|
|      A10|           Drake|     CA|   hip-hop|
|      A11|          Eminem|     US|   hip-hop|
|      A12|       Daft Punk|     FR|electronic|
|      A13|    David Guetta|     FR|electronic|
|      A14|   Calvin Harris|     GB|electronic|
|      A15|        Coldplay|     GB|      rock|
|      A16|  Arctic Monkeys|     GB|      rock|
|      A17| Imagine Dragons|     US|      rock|
|      A18|           Queen|     GB|    

### Exercise 7 (Stretch) — Lazy or not?
For each line below, decide **before running**: transformation or action? Then check with `explain()` on `result`.

```python
a = users.filter(F.col("plan") == "premium")
b = a.withColumn("age", 2026 - F.col("birth_year"))
c = b.count()
result = b.select("user_id", "age").orderBy("age")
result.show(3)
```

In [42]:
# YOUR CODE HERE

import time

# start = time.perf_counter()
# a = users.filter(F.col("plan") == "premium") big = spark.range(30_000_000).withColumn("squared", F.col("id") * F.col("id")).filter(F.col("id") % 7 == 0)
# print(f"transformations: {time.perf_counter() - start:.3f} s")  # nothing happened yet!

# start = time.perf_counter()
# print(big.count())
# print(f"action:          {time.perf_counter() - start:.3f} s")  # now Spark works

a = users.filter(F.col("plan") == "premium")
a.explain()

== Physical Plan ==
*(1) Filter (isnotnull(plan#56) AND (plan#56 = premium))
+- FileScan csv [user_id#53,signup_date#54,country#55,plan#56,birth_year#57] Batched: false, DataFilters: [isnotnull(plan#56), (plan#56 = premium)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/course/data/reference/users.csv], PartitionFilters: [], PushedFilters: [IsNotNull(plan), EqualTo(plan,premium)], ReadSchema: struct<user_id:string,signup_date:date,country:string,plan:string,birth_year:int>


